In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import gc
import time
import copy
import os
import random
import warnings
from joblib import Parallel, delayed
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 1000)
pd.set_option("display.max_rows", 1000)


In [ ]:
%run "TFT_Dataset_Pytorch.ipynb"

In [ ]:
%run "TFT_Models_Pytorch.ipynb"

In [ ]:
%run "TFT_Losses_Pytorch.ipynb"

In [ ]:
%run "TFT_Trainer_Pytorch.ipynb"

In [ ]:
os.environ['JOBLIB_TEMP_FOLDER'] = '/tmp'

In [ ]:
"""
# read in the datafiles -- assuming they are in the current directory
train_sales_df = pd.read_csv("./rohlik_dataset/sales_train.csv", parse_dates=['date'])
test_sales_df = pd.read_csv("./rohlik_dataset/sales_test.csv", parse_dates=['date'])
inventory_df = pd.read_csv("./rohlik_dataset/inventory.csv")
calendar_df = pd.read_csv("./rohlik_dataset/calendar.csv", parse_dates=['date'])
weights_df = pd.read_csv("./rohlik_dataset/test_weights.csv")

# merge the above datasets into a single dataset
# add 'availability' col to test_Sales & set it to null

test_sales_df['availability'] = np.nan
sales_df = pd.concat([train_sales_df, test_sales_df], axis=0)
sales_df = sales_df.sort_values(by=['unique_id','date'], ascending=True).reset_index(drop=True)

# merge inventory
sales_df = sales_df.merge(inventory_df, on=['unique_id','warehouse'], how='left')

# merge calendar
#calendar_df = pd.get_dummies(calendar_df, columns=['holiday_name'])
sales_df = sales_df.merge(calendar_df, on=['date','warehouse'], how='left')

# merge weights
sales_df = sales_df.merge(weights_df, on=['unique_id'], how='left')

# create date features
sales_df['weekday'] = sales_df['date'].dt.weekday
sales_df['week'] = sales_df['date'].dt.isocalendar().week
sales_df['month'] = sales_df['date'].dt.month
sales_df['year'] = sales_df['date'].dt.year
sales_df['prev_year'] = (sales_df['date'] - timedelta(days=1)).dt.year
sales_df['day'] = sales_df['date'].dt.day
sales_df['is_month_start'] = sales_df['date'].dt.is_month_start
sales_df['is_month_end'] = sales_df['date'].dt.is_month_end
sales_df['quarter'] = sales_df['date'].dt.quarter
sales_df['weekend']=(sales_df['weekday']>4).astype(np.int8)
sales_df['days_since_2020'] = (sales_df['date'] - pd.to_datetime('2020-01-01')).dt.days.astype('int')

# add 'country' as a feature
store2country = {
        'Budapest_1': 'Hungary',
        'Prague_2': 'Czechia',
        'Brno_1': 'Czechia',
        'Prague_1': 'Czechia',
        'Prague_3': 'Czechia',
        'Munich_1': 'Germany',
        'Frankfurt_1': 'Germany'
}

sales_df['country'] = sales_df['warehouse'].apply(lambda x:store2country[x])

# Basic fillna ['total_orders','sales','availability'] as mean of date
sales_df['sales'] = sales_df['sales'].fillna(sales_df.groupby(['unique_id','weekday'])['sales'].transform('mean'))
sales_df['total_orders'] = sales_df['total_orders'].fillna(sales_df.groupby(['unique_id','weekday'])['total_orders'].transform('mean'))
sales_df['availability'] = sales_df['availability'].fillna(sales_df.groupby(['unique_id','weekday'])['availability'].transform('mean'))
# fill 0 for any edge cases
sales_df['sales'] = sales_df['sales'].fillna(0)
sales_df['availability'] = sales_df['availability'].fillna(0)

# check for nulls in any columns
print(sales_df.columns[sales_df.isnull().any()])

# writeout the processed dataset
sales_df.to_csv("./rohlik_dataset/sales_processed_tft.csv", index=False)

"""

In [ ]:
df = pd.read_csv("../rohlik_dataset/sales_processed_tft.csv", dtype={'unique_id':'str'}, parse_dates=['date'])
df = df[df['year']>=2021]

# power transform target col using square root
df['sales'] = np.sqrt(df['sales'])

df['holiday_name'] = df['holiday_name'].fillna('None')

# check for nulls in any columns
print(df.columns[df.isnull().any()])

# the submission requires forecasts for only a subset of unique_ids; extract these ids for re-weighting purpose later
solution_df = pd.read_csv("../../rohlik-sales-forecasting-challenge-v2/solution.csv")
solution_df['unique_id'] = solution_df['id'].str.split('_').str[0]
all_solution_unique_ids = solution_df['unique_id'].unique().tolist()

print(len(all_solution_unique_ids)) # shows 3625

# create entity weights
def create_weights(df):
    max_test_wt = df[df['unique_id'].isin(all_solution_unique_ids)]['weight'].unique().max()
    q25_test_wt = np.quantile(df['weight'].unique(), q=0.25)
    df['weight'] = np.clip(df['weight'], a_min=q25_test_wt, a_max=max_test_wt)
    return df

df = create_weights(df)


In [ ]:
df.head()

In [ ]:
# feature groups
date_features = ['weekday','week','month','day','quarter','year','prev_year','is_month_start','is_month_end','weekend']
other_features = ['total_orders', 'sell_price_main']
promo_features = ['type_0_discount',
                  'type_1_discount',
                  'type_2_discount',
                  'type_3_discount',
                  'type_4_discount',
                  'type_5_discount',
                  'type_6_discount']

event_features = ['holiday',
                  'shops_closed', 
                  'winter_school_holidays',
                  'school_holidays',
                  'holiday_name']
                  
static_categorical_features = ['name',
                               'country',
                               'warehouse',
                               'L1_category_name_en',
                               'L2_category_name_en',
                               'L3_category_name_en',
                               'L4_category_name_en']

# config dict keys
id_col = 'unique_id'
datetime_col = 'date'
target_col = 'sales'
static_numeric_col_list = []
static_categorical_col_list = static_categorical_features
temporal_known_numeric_col_list = other_features + promo_features
temporal_known_categorical_col_list = date_features + event_features
temporal_unknown_numeric_col_list = ['availability']
temporal_unknown_categorical_col_list = []
weight_col = 'weight'

# Config dict for dataset
features_config = {
    "entity_col": id_col,          
    "time_index_col": datetime_col,         
    "target_col": target_col,
    "static_numeric_col_list": static_numeric_col_list,
    "static_categorical_col_list": static_categorical_col_list,
    "temporal_known_numeric_col_list": temporal_known_numeric_col_list,
    "temporal_known_categorical_col_list": temporal_known_categorical_col_list,
    "temporal_unknown_numeric_col_list": temporal_unknown_numeric_col_list,
    "temporal_unknown_categorical_col_list": temporal_unknown_categorical_col_list,
    "wt_col": weight_col
}


In [ ]:
# Other dataset Parameters

# context window length
historical_steps=60
forecast_steps=14
train_min_historical_steps=14
test_min_historical_steps=14
infer_min_historical_steps=0

# no. of samples
train_stride=5
test_stride=1

# scaler and encoder location
scaler_path='./rohlik_experiment/train_scalers.pkl'
scaling_strategy='entity_level'
scaling_method='mean'
encoders_path='./rohlik_experiment'

# parallel processing
train_n_jobs = 6
test_n_jobs = 4
infer_n_jobs = 2

# cutoff periods for train/val/infer
train_start = pd.to_datetime('2021-01-01', format="%Y-%m-%d")
train_cutoff = pd.to_datetime('2024-04-30', format="%Y-%m-%d")

test_start = train_cutoff - timedelta(days=historical_steps)
test_cutoff = pd.to_datetime('2024-06-02', format="%Y-%m-%d")

infer_cutoff = pd.to_datetime('2024-06-16', format="%Y-%m-%d")
infer_start = infer_cutoff - timedelta(days=historical_steps + forecast_steps)

print(f" train_start: {train_start}, train_cutoff: {train_cutoff}")
print(f" test_start: {test_start}, test_cutoff: {test_cutoff}")
print(f" infer_start: {infer_start}, infer_cutoff: {infer_cutoff}")

# create train & test datasets
df = df.sort_values(by=['unique_id','date'], ascending=True)

train_df = df[df['date'] <= train_cutoff].copy()
test_df = df[(df['date'] > test_start) & (df['date'] <= test_cutoff)].copy()

# get last historical_steps + forecast_steps steps for inference
#infer_df = df[(df['date'] > infer_start) & (df['date'] <= infer_cutoff)].copy()

infer_df = df[df['date'] <= infer_cutoff].copy()
infer_df =  infer_df.groupby(['unique_id'], group_keys=False).tail(historical_steps + forecast_steps)


In [ ]:
# Create datasets

train_dataset = OptimizedTFTDataset(
                data_source=train_df,
                features_config=features_config, 
                historical_steps=historical_steps,
                prediction_steps=forecast_steps,
                stride=train_stride,
                enable_padding=True,
                padding_strategy = 'zero',
                categorical_padding_value = -1,
                min_historical_steps=train_min_historical_steps,
                allow_inference_only_entities=False,
                scaler_path=scaler_path,
                scaling_strategy=scaling_strategy,
                scaling_method=scaling_method,
                cold_start_scaler_cols=None,
                mean_scaler_epsilon=1.0,
                recency_alpha=1.0,
                n_jobs=train_n_jobs,
                max_series=None,
                mode='train',
                encoders_path=encoders_path,
                fit_encoders=None,
                preprocessing_fn=None)

test_dataset = OptimizedTFTDataset(
                data_source=test_df,
                features_config=features_config, 
                historical_steps=historical_steps,
                prediction_steps=forecast_steps,
                stride=test_stride,
                enable_padding=True,
                padding_strategy = 'zero',
                categorical_padding_value = -1,
                min_historical_steps=test_min_historical_steps,
                allow_inference_only_entities=False,
                scaler_path=scaler_path,
                scaling_strategy=scaling_strategy,
                scaling_method=scaling_method,
                cold_start_scaler_cols=None,
                mean_scaler_epsilon=1.0,
                recency_alpha=0,
                n_jobs=test_n_jobs,
                max_series=None,
                mode='test',
                encoders_path=encoders_path,
                fit_encoders=None,
                preprocessing_fn=None)

infer_dataset = OptimizedTFTDataset(
                data_source=infer_df,
                features_config=features_config, 
                historical_steps=historical_steps,
                prediction_steps=forecast_steps,
                stride=test_stride,
                enable_padding=True,
                padding_strategy = 'zero',
                categorical_padding_value = -1,
                min_historical_steps=infer_min_historical_steps,
                allow_inference_only_entities=True,
                scaler_path=scaler_path,
                scaling_strategy=scaling_strategy,
                scaling_method=scaling_method,
                cold_start_scaler_cols=['name','country'], 
                mean_scaler_epsilon=1.0,
                recency_alpha=0,
                n_jobs=1,
                max_series=None,
                mode='test',
                encoders_path=encoders_path,
                fit_encoders=None,
                preprocessing_fn=None)

# create dataloaders
#train_dataloader = DataLoader(train_dataset, batch_size=512, shuffle=True, pin_memory=True, drop_last=False)
#test_dataloader = DataLoader(test_dataset, batch_size=512, shuffle=False, pin_memory=True, drop_last=False)
#infer_dataloader = DataLoader(infer_dataset, batch_size=512, shuffle=False, pin_memory=True, drop_last=False)

# create TFTAdapters
#train_adapter = TFTDataAdapter(train_dataset)
#test_adapter = TFTDataAdapter(test_dataset)
#infer_adapter = TFTDataAdapter(infer_dataset)


# create dataloaders
train_dataloader, train_adapter = create_tft_dataloader(train_dataset, 
                                                        batch_size=512, 
                                                        shuffle=True,
                                                        num_workers=0,
                                                        drop_last=False,
                                                        pin_memory=True)

test_dataloader, test_adapter = create_tft_dataloader(test_dataset, 
                                                      batch_size=512, 
                                                      shuffle=False,
                                                      num_workers=0,
                                                      drop_last=False,
                                                      pin_memory=True)

infer_dataloader, infer_adapter = create_tft_dataloader(infer_dataset,
                                                        batch_size=512,
                                                        shuffle=False,
                                                        num_workers=0,
                                                        drop_last=False,
                                                        pin_memory=True)

In [ ]:
#prefetch_train_loader = PrefetchDataLoader(train_dataset,
#                                         batch_size=256,
#                                         shuffle=True,
#                                         collate_fn=None,
#                                         prefetch_size=4,
#                                         device='cuda',
#                                         pin_memory=True)


In [ ]:
#%%timeit
#device = torch.device("cuda")
#for batch_idx, batch in enumerate(train_dataloader):
#    batch_gpu = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
#    print(batch_idx)

In [ ]:
#%%timeit
#for batch_idx, batch in enumerate(prefetch_train_loader):
#    print(batch_idx)

In [ ]:
# sample dataset outputs

#static_categorical: Optional[List[torch.Tensor]] = None,
#static_continuous: Optional[List[torch.Tensor]] = None,
#historical_categorical: Optional[List[torch.Tensor]] = None,
#historical_continuous: Optional[List[torch.Tensor]] = None,
#future_categorical: Optional[List[torch.Tensor]] = None,
#future_continuous: Optional[List[torch.Tensor]] = None,
#padding_mask: Optional[torch.Tensor] = None

for batch in infer_dataloader:
    model_inputs = infer_adapter.adapt_for_tft(batch)
    print("Model Input Keys: \n")
    for k, v in model_inputs.items():
        if isinstance(v, list):
            print(f"Key: {k}: type: List, num_elements: {len(v)}, shape: {v[0].shape}")
        else:
            print(f"Key: {k}: type: Tensor, shape: {v.shape}")
    break
    

In [ ]:
#for batch_idx, batch in enumerate(infer_dataloader):
#    print(batch_idx," : \n", np.unique(batch['entity_id']), "\n", np.unique(batch['window_idx']))

In [ ]:
#sample_batch = next(iter(test_dataloader))

In [ ]:
#np.unique(sample_batch['entity_id'])

In [ ]:
# create Model

# first create embedding dims dict for various categorical variables
categorical_embedding_dims = create_uniform_embedding_dims(train_dataset, hidden_layer_size=160)

print(categorical_embedding_dims)

model = TemporalFusionTransformer(
        # Model architecture parameters
        hidden_layer_size = 160,
        num_attention_heads = 1,
        num_lstm_layers = 1,
        num_attention_layers = 4,
        dropout_rate = 0.1,
        
        # Input specification
        num_static_categorical = 7,
        num_static_continuous = 0,
        num_historical_categorical = 15,
        num_historical_continuous = 11,
        num_future_categorical = 15,
        num_future_continuous = 9,
    
        # embedding dims for categorical variables
        categorical_embedding_dims = categorical_embedding_dims,
        
        # Time dimensions
        historical_steps = historical_steps,
        prediction_steps = forecast_steps,
        
        # Output specification
        num_outputs = 1,
        device = 'cuda' #'cuda' if torch.cuda.is_available() else 'cpu'
    )


In [ ]:
# Create Trainer

trainer = TFTTrainer(model = model,
                     # data params
                     train_loader = train_dataloader,
                     val_loader = test_dataloader,
                     train_adapter = train_adapter,
                     val_adapater = test_adapter,
                     # loss params
                     loss_type = 'huber',
                     loss_params = {'delta': 0.75},
                     # optimizer params
                     optimizer_type = 'adam',
                     learning_rate = 0.0001,
                     weight_decay = 1e-5,
                     momentum = 0.9,  # For SGD
                     # Scheduler configuration
                     scheduler_type = 'reduce_on_plateau',
                     scheduler_factor = 0.5,
                     scheduler_patience = 5,
                     scheduler_t0 = 10,  # For cosine annealing
                     scheduler_t_mult = 2,  # For cosine annealing
                     # Training options
                     enable_gradient_clipping = True,
                     max_grad_norm = 1.0,
                     enable_train_sample_weighting = False,
                     enable_val_sample_weighting = False,
                     # Mixed Precision Training
                     enable_mixed_precision = False,
                     # Checkpointing
                     save_path = './rohlik_experiment/model',
                     save_every = 1)


In [ ]:
trainer.train(num_epochs=50, patience=5)

In [ ]:
inferencewithtracking = TFTInferenceWithTracking(model_path = './rohlik_experiment/model/best_model.pt', 
                                                 model = model,
                                                 adapter = infer_adapter,
                                                 device = 'cuda')


In [ ]:
results_df = inferencewithtracking.predict_with_metadata(infer_dataloader)

In [ ]:
results_df.head()

In [ ]:
results_df.shape

In [ ]:
solution_df.shape

In [ ]:
solution_df.head()

In [ ]:
results_df['id'] = results_df['entity_id'].astype(str) + '_' + results_df['timestamp'].astype(str)


In [ ]:
submit_df = solution_df.merge(results_df, on=['id'], how='inner')

In [ ]:
submit_df.shape

In [ ]:
submit_df.head()

In [ ]:
submit_df.drop(columns=['sales_hat','unique_id','entity_id','timestamp','window_idx','horizon','actual_sales'], inplace=True)
submit_df.rename(columns={'pred_0':'sales_hat'}, inplace=True)


In [ ]:
submit_df.head()

In [ ]:
submit_df['sales_hat'] = np.square(submit_df['sales_hat'])

In [ ]:
submit_df.head()

In [ ]:
submit_df.to_csv("./rohlik_experiment/output/kg_submission_tft_huber75_wtd_val.csv", index=False)